# stage 2: Hidden test evaluation

Loads stage 1 from checkpoint and predicts on the hidden test values.

In [1]:
import pandas as pd
import numpy as np
import json
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from tqdm.auto import tqdm

cfg = json.load(open("model_checkpoint/config.json"))

c:\Users\dleon\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = AutoTokenizer.from_pretrained(cfg["encoder"])
model = AutoModel.from_pretrained(cfg["encoder"])
model.eval()

def chunk_text(text, max_length=512, overlap=0):
    token_id = tokenizer(text, add_special_tokens=False)["input_ids"]
    body_size = max_length - 2
    step = body_size - overlap
    chunks = []
    for i in range(0, len(token_id), step):
        body = token_id[i:i + body_size]
        chunks.append([tokenizer.cls_token_id] + body + [tokenizer.sep_token_id])
        if i + body_size >= len(token_id):
            break
    return chunks

@torch.no_grad()
def get_embeddings(text, overlap=0):
    chunks = chunk_text(text, overlap=overlap)
    vectors = []
    for c in chunks:
        outputs = model(torch.tensor([c]))
        vectors.append(outputs.last_hidden_state[:, 0, :].squeeze(0))
    return torch.stack(vectors).mean(dim=0).numpy()

@torch.no_grad()
def head_probs(head, X):
    logits = head(torch.tensor(X, dtype=torch.float32))
    return torch.softmax(logits, dim=1)[:, 1].numpy()

def evaluate_model(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    print(f"Accuracy: {acc:.4f}")
    print(f"Balanced Accuracy: {bal_acc:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    return acc, bal_acc, cm

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9336.44it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
head = nn.Linear(cfg["hidden_dim"], 2)
head.load_state_dict(torch.load("model_checkpoint/head.pt"))
head.eval()
print("Model loaded successfully.")

Model loaded successfully.


In [4]:
hidden = pd.read_csv("data/hidden_test_with_labels.csv")
print(f"Loaded hidden test data with shape: {hidden.shape}")
print(hidden.columns.tolist())
print(hidden['label'].value_counts())

Loaded hidden test data with shape: (600, 5)
['id', 'text', 'label', 'label_name', 'source_file']
label
0    300
1    300
Name: count, dtype: int64


## Results

Accuracy achieved: 0.7567, balanced accuracy = 0.7567

| | Predicted 0 | Predicted 1 |
|----|----|----|
| **Actual 0** | 203 | 97 |
| **Actual 1** | 49 | 251 |

The hidden set had 600 reviews, with an even split of each class, 300 per. Negative
recall was 0.677 and positive recall was 0.837.

## Comparison

| | CV (train) | Public test | Hidden test |
|---|---|---|---|
| Examples | 240 | 400 | 600 |
| Balanced accuracy | 0.7889 | 0.7650 | 0.7567 |

The hidden test results came out to be 0.83 points under the public test results and 3 under the CV estimate. What we can observe from this is that it generalizes decently well, and the CV wasn't massively inflated. The model had a similar error pattern, in which it identified positives (public = 0.825, hidden = 0.837) better than negatives (public = 0.705, hidden = 0.677). That gap comes from training on 180 positives and only 60 negatives. Weighted cross entropy narrowed it but didn't close it.

The predicted labels on the hidden test set came out to be 348 positives and 252 negatives against a 300/300 split.

In [5]:
x_hidden = np.array([get_embeddings(t, overlap=cfg["overlap"]) for t in tqdm(hidden['text'], desc="hidden")])
y_hidden = hidden['label'].values

hidden_probs = head_probs(head, x_hidden)
hidden_pred = (hidden_probs >= cfg["threshold"]).astype(int)

print("Evaluation on hidden test set:")
_= evaluate_model(y_hidden, hidden_pred)

hidden: 100%|██████████| 600/600 [01:57<00:00,  5.10it/s]

Evaluation on hidden test set:
Accuracy: 0.7567
Balanced Accuracy: 0.7567
Confusion Matrix:
[[203  97]
 [ 49 251]]


In [6]:
pd.DataFrame({"id": hidden['id'], "predicted_label": hidden_pred}).to_csv("hidden_test_predictions.csv", index=False)

check = pd.read_csv("hidden_test_predictions.csv")
print(check.shape, check.columns.tolist())
print(check["predicted_label"].value_counts())

(600, 2) ['id', 'predicted_label']
predicted_label
1    348
0    252
Name: count, dtype: int64


## What I would do next

The first thing I would want to do is unfreeze some of the layers and compare them to the frozen version. Since the encoder stayed frozen, the representations were never adapted to sentiment and only the head's parameters were trained. I avoided doing this here due to the time constraint and wanting something that could be evaluated as soon as possible.

Secondly, I'd like to train with more negative reviews. The recall gap discussed above stayed consistent across both test sets, which means it's more than likely not noise, but left over from only having 60 negatives. Weighted cross entropy helped, however it can't add info that isn't in the data.

Lastly, I'd use weighted chunks instead of the average of the chunks. As it stands every chunk is equal, but reviews often have the verdict towards the end. This means in a 6 chunk review that verdict only counts as 1/6 of the vector. So I'd like to try weighting the later chunks more, or even learning attention over the chunks.